## Modélisation des chiffres d'affaires

### 1. Chargement des données

Chargement des jeux d'entraînement et de test préparés lors du feature engineering.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

DATA_DIR = Path("../data/Train_Test")

X_train = pd.read_csv(DATA_DIR / "X_train.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv")

X_test = pd.read_csv(DATA_DIR / "X_test.csv")
y_test = pd.read_csv(DATA_DIR / "y_test.csv")

print("Dimensions des données :")
print("X_train :", X_train.shape)
print("y_train :", y_train.shape)
print("X_test  :", X_test.shape)
print("y_test  :", y_test.shape)

Dimensions des données :
X_train : (578030, 25)
y_train : (578030, 1)
X_test  : (145032, 25)
y_test  : (145032, 1)


In [ ]:
# controle de la qualité des données

print("Valeurs manquantes :")
print("X_train :", X_train.isna().sum().sum())
print("y_train :", y_train.isna().sum().sum())


print("\nValeurs infinies :")
print("X_train :", np.isinf(X_train.select_dtypes(include=np.number)).sum().sum())

print("\nStatistiques de la cible :")
print(y_train.describe())

Valeurs manquantes :
X_train : 0
y_train : 0

Valeurs infinies :
X_train : 0

Statistiques de la cible :
               Sales
count  578030.000000
mean     6866.063014
std      2323.776407
min      2574.000000
25%      5083.000000
50%      6671.000000
75%      8334.000000
max     21387.000000


### 2. Préparation de la validation temporelle

Les données étant organisées par date et par magasin, les séparations temporelles sont effectuées par journée complète afin d'éviter les mélanges entre entraînement et validation.

In [4]:
from sklearn.model_selection import TimeSeriesSplit

def reconstruire_dates(X):
    return pd.to_datetime({
        "year": X["Year"],
        "month": X["Month"],
        "day": X["Day"]
    })

dates_train = reconstruire_dates(X_train)
dates_test = reconstruire_dates(X_test)

print("Période d'entraînement :")
print(dates_train.min(), "→", dates_train.max())

print("\nPériode de test :")
print(dates_test.min(), "→", dates_test.max())

print("\nNombre de dates d'entraînement :", dates_train.nunique())
print("Nombre de dates de test :", dates_test.nunique())

print("\nEntraînement chronologique :", dates_train.is_monotonic_increasing)
print("Test chronologique :", dates_test.is_monotonic_increasing)
print("Séparation temporelle respectée :", dates_train.max() < dates_test.min())

Période d'entraînement :
2013-01-01 00:00:00 → 2014-07-31 00:00:00

Période de test :
2014-08-01 00:00:00 → 2014-12-23 00:00:00

Nombre de dates d'entraînement : 577
Nombre de dates de test : 145

Entraînement chronologique : True
Test chronologique : True
Séparation temporelle respectée : True


### 3. Création des folds temporels

Création de trois découpages temporels à fenêtre croissante avec `TimeSeriesSplit`.

Les dates sont séparées avant de retrouver les lignes correspondantes afin de conserver tous les magasins d'une même journée dans le même ensemble.

Ces folds seront réutilisés pour entraîner et comparer les modèles sur les mêmes périodes de validation.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

# récupérer les dates uniques dans l'ordre chronologique
dates_uniques = np.sort(dates_train.unique())

# créer trois découpages temporels
tscv = TimeSeriesSplit(n_splits=3)

folds = []

for train_idx, val_idx in tscv.split(dates_uniques):

    dates_fold_train = dates_uniques[train_idx]
    dates_fold_val = dates_uniques[val_idx]

    lignes_train = np.flatnonzero(
        dates_train.isin(dates_fold_train)
    )

    lignes_val = np.flatnonzero(
        dates_train.isin(dates_fold_val)
    )

    folds.append((lignes_train, lignes_val))

print("Nombre de folds créés :", len(folds))

Nombre de folds créés : 3


#### 3.1. Vérification des folds temporels

Vérification des dimensions et des périodes de chaque fold.

Chaque fold doit respecter trois conditions :
- Les dates d'entraînement précèdent les dates de validation.
- Toutes les observations d'une même journée restent dans le même ensemble.
- Les périodes d'entraînement augmentent progressivement.

In [8]:
for i, (train_idx, val_idx) in enumerate(folds, start=1):

    dates_fold_train = dates_train.iloc[train_idx]
    dates_fold_val = dates_train.iloc[val_idx]

    print(f"\nFold {i}")
    print("-" * 40)

    print("Lignes d'entraînement :", len(train_idx))
    print("Lignes de validation  :", len(val_idx))

    print(
        "Période d'entraînement :",
        dates_fold_train.min().date(),
        "→",
        dates_fold_train.max().date()
    )

    print(
        "Période de validation :",
        dates_fold_val.min().date(),
        "→",
        dates_fold_val.max().date()
    )

    print(
        "Séparation temporelle respectée :",
        dates_fold_train.max() < dates_fold_val.min()
    )

    print(
        "Dates communes entre les deux ensembles :",
        len(set(dates_fold_train.unique()) & set(dates_fold_val.unique()))
    )


Fold 1
----------------------------------------
Lignes d'entraînement : 145844
Lignes de validation  : 143927
Période d'entraînement : 2013-01-01 → 2013-05-25
Période de validation : 2013-05-26 → 2013-10-16
Séparation temporelle respectée : True
Dates communes entre les deux ensembles : 0

Fold 2
----------------------------------------
Lignes d'entraînement : 289771
Lignes de validation  : 143759
Période d'entraînement : 2013-01-01 → 2013-10-16
Période de validation : 2013-10-17 → 2014-03-09
Séparation temporelle respectée : True
Dates communes entre les deux ensembles : 0

Fold 3
----------------------------------------
Lignes d'entraînement : 433530
Lignes de validation  : 144500
Période d'entraînement : 2013-01-01 → 2014-03-09
Période de validation : 2014-03-10 → 2014-07-31
Séparation temporelle respectée : True
Dates communes entre les deux ensembles : 0


#### 3.2. Préparation des données de validation

Préparation des jeux de données d'entraînement et de validation pour chaque fold.

Les indices calculés précédemment permettent de sélectionner les observations correspondantes dans `X_train` et `y_train`.

Les données de test restent totalement exclues de cette étape.

In [9]:
donnees_folds = []

for train_idx, val_idx in folds:

    X_fold_train = X_train.iloc[train_idx]
    y_fold_train = y_train.iloc[train_idx].squeeze()

    X_fold_val = X_train.iloc[val_idx]
    y_fold_val = y_train.iloc[val_idx].squeeze()

    donnees_folds.append({
        "X_train": X_fold_train,
        "y_train": y_fold_train,
        "X_val": X_fold_val,
        "y_val": y_fold_val
    })

print("Nombre de jeux de données préparés :", len(donnees_folds))

for i, fold in enumerate(donnees_folds, start=1):
    print(f"\nFold {i}")
    print("X_train :", fold["X_train"].shape)
    print("y_train :", fold["y_train"].shape)
    print("X_val   :", fold["X_val"].shape)
    print("y_val   :", fold["y_val"].shape)

Nombre de jeux de données préparés : 3

Fold 1
X_train : (145844, 25)
y_train : (145844,)
X_val   : (143927, 25)
y_val   : (143927,)

Fold 2
X_train : (289771, 25)
y_train : (289771,)
X_val   : (143759, 25)
y_val   : (143759,)

Fold 3
X_train : (433530, 25)
y_train : (433530,)
X_val   : (144500, 25)
y_val   : (144500,)


### 4. Modèle de référence (Baseline)

Construction d'un modèle de référence basé sur la moyenne des chiffres d'affaires.

Ce modèle prédit une valeur constante pour toutes les observations.

Il servira de référence pour évaluer les performances des modèles de Machine Learning.

In [10]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

resultats_baseline = []

for i, fold in enumerate(donnees_folds, start=1):

    modele = DummyRegressor(strategy="mean")

    modele.fit(
        fold["X_train"],
        fold["y_train"]
    )

    predictions = modele.predict(fold["X_val"])

    mae = mean_absolute_error(
        fold["y_val"],
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            fold["y_val"],
            predictions
        )
    )

    resultats_baseline.append({
        "Fold": i,
        "MAE": mae,
        "RMSE": rmse
    })

    print(f"Fold {i} - MAE : {mae:.2f} | RMSE : {rmse:.2f}")

print("\nRésultats moyens du baseline :")

print(
    pd.DataFrame(resultats_baseline)[["MAE", "RMSE"]].mean()
)

Fold 1 - MAE : 1783.45 | RMSE : 2172.23
Fold 2 - MAE : 2075.27 | RMSE : 2733.80
Fold 3 - MAE : 1799.30 | RMSE : 2182.65

Résultats moyens du baseline :
MAE     1886.006548
RMSE    2362.891400
dtype: float64


### 5. Modèle Random Forest

Entraînement d'un modèle Random Forest Regressor sur les trois folds temporels.

Le modèle utilise les variables explicatives non normalisées, car les arbres de décision ne nécessitent pas de standardisation.

Les performances sont mesurées avec le MAE et le RMSE sur chaque période de validation.

In [11]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

resultats_random_forest = []

for i, fold in enumerate(donnees_folds, start=1):

    modele = RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_split=2,
        min_samples_leaf=2,
        max_features=1.0,
        random_state=42,
        n_jobs=-1
    )

    modele.fit(
        fold["X_train"],
        fold["y_train"]
    )

    predictions = modele.predict(fold["X_val"])

    mae = mean_absolute_error(
        fold["y_val"],
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            fold["y_val"],
            predictions
        )
    )

    resultats_random_forest.append({
        "Fold": i,
        "MAE": mae,
        "RMSE": rmse
    })

    print(f"Fold {i} - MAE : {mae:.2f} | RMSE : {rmse:.2f}")

print("\nRésultats moyens du Random Forest :")

print(
    pd.DataFrame(resultats_random_forest)[["MAE", "RMSE"]].mean()
)

Fold 1 - MAE : 890.77 | RMSE : 1165.93
Fold 2 - MAE : 1215.55 | RMSE : 1885.72
Fold 3 - MAE : 917.75 | RMSE : 1209.15

Résultats moyens du Random Forest :
MAE     1008.022637
RMSE    1420.266906
dtype: float64


#### 5.1. Analyse des résultats du Random Forest

Le modèle Random Forest obtient les performances moyennes suivantes sur les trois folds temporels :

- MAE moyen : 1 008,02
- RMSE moyen : 1 420,27
- Temps d'entraînement : 11 min 18,3 s → sur pc 16 go ram

Le fold 2 présente des erreurs plus importantes (MAE : 1 215,55 ; RMSE : 1 885,72) que les folds 1 et 3. Cela peut indiquer une difficulté du modèle à généraliser sur cette période particulière.

Le modèle constitue une première référence intéressante. Ses performances devront être comparées à celles du baseline et des autres modèles avant de déterminer le meilleur candidat.

Les paramètres utilisés sont une configuration initiale et pourront être optimisés ultérieurement.

In [12]:
tableau_random_forest = pd.DataFrame(resultats_random_forest)

display(tableau_random_forest.round(2))

print("\nMoyennes des performances :")
display(
    tableau_random_forest[["MAE", "RMSE"]]
    .mean()
    .round(2)
    .to_frame("Moyenne")
)

,Fold,MAE,RMSE
0,1,890.77,1165.93
1,2,1215.55,1885.72
2,3,917.75,1209.15



Moyennes des performances :


,Moyenne
MAE,1008.02
RMSE,1420.27
